# 🚇 GIS-Based Transit Accessibility, Urban Catchment and Transit-Oriented Development (TOD) Potential Assessment of Chennai Metro Rail Stations

## 📌 Project Overview
This mini project evaluates **where Chennai Metro stations are, whom they can reach, and how ready their surroundings are for Transit-Oriented Development**. It combines GIS operations (projection, buffering, overlap analysis, distance surfaces, spatial autocorrelation) with a transparent multi-criteria **TOD Potential Index** built on the *5D framework* (Density, Diversity, Design, Destination accessibility, Distance to transit).

It extends the CMRL ridership notebook (same datasets) from *"how many people board?"* to *"how accessible is each station, how good is its catchment, and where should TOD policy go first?"*

### 🎯 Research Questions
1. **Network geometry** – How are stations spaced, how much of the city do they cover at 400 m / 800 m / 1.2 km walking distances, and what does Phase 2 add?
2. **Urban catchment quality** – How dense, mixed-use and walkable are station catchments (Euclidean vs. street-network catchment)?
3. **Accessibility** – Which stations are best placed to reach activities and population (Hansen gravity accessibility)?
4. **TOD potential** – Which stations score highest on a composite 5D index, and how *stable* is that ranking under different weights?
5. **Node–Place balance** – Which stations are under-built for their transit connectivity (intensify) vs. under-served for their urban intensity (improve access)?
6. **Spatial pattern** – Are high/low TOD stations spatially clustered (Moran's I, Getis-Ord Gi*)?
7. **Validation** – Does the index explain observed weekday ridership of operational stations, and which stations over/under-perform?
8. **Phase 2 priorities** – Which planned stations should get TOD attention first?

### 🗂️ Data (same package as the CMRL ridership notebook)
| File | Used for |
|---|---|
| `chennai_metro_stations.csv` | Station directory, coordinates, line, phase, layout |
| `chennai_station_catchment_features.csv` | 5D built-environment features (population, built volume, intersections, POIs, bus stops, centrality …) |
| `chennai_metro_alignments.geojson` | Corridor centre-lines (mapping) |
| `cmrl_station_summary.csv` | Observed weekday/weekend boardings (validation only) |
| `chennai_metro_ridership_predictions.csv` | Optional – benchmark ML predictions for comparison |

### 🧭 Methodology Pipeline
```
Load & validate  →  Project to UTM 44N (EPSG:32644)  →  Network geometry (NN distance, buffers, overlap)
      →  Coverage surface (250 m grid)  →  Hansen accessibility  →  Urban-form indicators
      →  5D TOD Index (percentile scoring, 3 weight schemes, Monte-Carlo stability)
      →  Node–Place model + K-Means typology  →  Spatial statistics (Moran's I, Gi*)
      →  Ridership validation (Spearman, Ridge LOO-CV)  →  Phase 2 priority list  →  Maps, dashboards, export
```
> **Note on scope:** catchments are *straight-line radii* from the dataset; the street-network catchment area is used only as a walkability-efficiency indicator. All index scores are *relative* (percentile-based), not absolute.

## 1️⃣ Setup: Libraries & Plot Configuration

In [ ]:
import os, re, json, math, difflib, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from scipy import stats
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, r2_score, mean_absolute_error
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import LeaveOneOut, cross_val_predict

try:
    import geopandas as gpd
    import shapely
    from shapely.geometry import Point, LineString
    from shapely.ops import unary_union
    import folium
    from folium.plugins import HeatMap
except ImportError:
    import subprocess, sys
    print('Installing geopandas, shapely, folium ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'geopandas', 'shapely', 'folium'])
    import geopandas as gpd
    import shapely
    from shapely.geometry import Point, LineString
    from shapely.ops import unary_union
    import folium
    from folium.plugins import HeatMap

sns.set_theme(style='whitegrid', context='notebook', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 150, 'savefig.bbox': 'tight',
                     'axes.titleweight': 'bold', 'axes.titlesize': 12})

CRS_WGS84 = 'EPSG:4326'
CRS_UTM   = 'EPSG:32644'   # UTM zone 44N: metric CRS for Chennai (80.2 E)
RNG_SEED  = 42
print('Libraries ready | geopandas', gpd.__version__, '| shapely', shapely.__version__)

## 2️⃣ Load Data
The loader looks for the dataset in Colab's `/content`, Drive, or the current folder. If nothing is found in Colab it asks you to upload the files.

In [ ]:
CANDIDATE_DIRS = ['/content', '/content/drive/MyDrive', '.', './data', '/mnt/user-data/uploads']
DATA_DIR = next((d for d in CANDIDATE_DIRS
                 if os.path.exists(os.path.join(d, 'chennai_metro_stations.csv'))), None)

if DATA_DIR is None:
    try:
        from google.colab import files
        print('Dataset not found - please upload the CSV / GeoJSON files.')
        files.upload()
        DATA_DIR = '.'
    except ImportError:
        raise FileNotFoundError('Put the CMRL dataset files next to this notebook (or in /content).')

P = lambda name: os.path.join(DATA_DIR, name)
OUT_DIR = os.path.join(DATA_DIR if os.access(DATA_DIR, os.W_OK) else '.', 'outputs')
FIG_DIR = os.path.join(OUT_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
print('Data folder   :', os.path.abspath(DATA_DIR))
print('Output folder :', os.path.abspath(OUT_DIR))

stations   = pd.read_csv(P('chennai_metro_stations.csv'))
catch      = pd.read_csv(P('chennai_station_catchment_features.csv'))
alignments = gpd.read_file(P('chennai_metro_alignments.geojson'))
alignments = alignments[~alignments.geometry.to_wkt().duplicated()].reset_index(drop=True)   # source file contains duplicate segments
summary    = pd.read_csv(P('cmrl_station_summary.csv')) if os.path.exists(P('cmrl_station_summary.csv')) else None
pred_file  = pd.read_csv(P('chennai_metro_ridership_predictions.csv')) if os.path.exists(P('chennai_metro_ridership_predictions.csv')) else None

# ---- schema validation ----
need_st = {'station_id', 'name', 'line', 'phase', 'latitude', 'longitude'}
need_ca = {'station_id', 'catchment_area_km2', 'population', 'population_density', 'poi_total', 'bus_stops_500m'}
assert need_st <= set(stations.columns), f'Missing station columns: {need_st - set(stations.columns)}'
assert need_ca <= set(catch.columns),    f'Missing catchment columns: {need_ca - set(catch.columns)}'

for nm, d in [('stations', stations), ('catchment features', catch), ('alignments', alignments),
              ('station summary', summary), ('ridership predictions', pred_file)]:
    print(f'{nm:24s}', 'not found (optional)' if d is None else f'{d.shape[0]} rows x {d.shape[1]} cols')

## 3️⃣ Data Preparation & Spatial Setup
* Merge directory + catchment features on `station_id`.
* Flag **operational (Phase 1)** vs **Phase 2** stations and stations whose coordinates are flagged unreliable.
* Convert to a **metric CRS (UTM 44N)** – required for correct distances, buffers and areas (degrees are not metres).

In [ ]:
drop_dup = [c for c in ['name', 'phase', 'confidence', 'position_source', 'Line', 'Layout'] if c in catch.columns]
df = stations.merge(catch.drop(columns=drop_dup), on='station_id', how='left', validate='one_to_one')

# numeric coercion + gentle imputation (only the network-centrality fields have gaps in the source data)
num_cols = [c for c in df.columns if c not in stations.columns and c != 'city']
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')
imputed = {c: int(df[c].isna().sum()) for c in num_cols if df[c].isna().any()}
for c in imputed:
    df[c] = df[c].fillna(df[c].median())
print('Median-imputed values:', imputed if imputed else 'none')

df['line'] = df['line'].astype(str).str.strip()
df['operational'] = df['phase'].astype(str).str.contains('1')
df['status'] = np.where(df['operational'], 'Operational (Phase 1)', 'Phase 2')

rel = df['position_reliable'].astype(str).str.strip().str.lower() if 'position_reliable' in df else pd.Series('true', index=df.index)
df['pos_ok'] = ~rel.isin(['false', '0', 'no', 'n', 'f', 'approximate', 'low'])

def line_color(name):
    n = str(name).lower()
    for key, col in [('blue', '#1f5fbf'), ('green', '#2ca02c'), ('purple', '#7b3fa0'),
                     ('orange', '#f28e2b'), ('red', '#d62728'), ('yellow', '#e6b800')]:
        if key in n:
            return col
    return '#7f7f7f'
df['line_color'] = df['line'].map(line_color)

gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['longitude'], df['latitude']), crs=CRS_WGS84)
gdf_m = gdf.to_crs(CRS_UTM).reset_index(drop=True)
align_m = alignments.to_crs(CRS_UTM)
align_m['is_planned'] = align_m['construction'].notna() if 'construction' in align_m else False

XY = np.c_[gdf_m.geometry.x, gdf_m.geometry.y]
D  = cdist(XY, XY)                       # full station-to-station straight-line distance matrix (m)
N  = len(gdf_m)

# catchment radius implied by the dataset's catchment area (A = pi r^2)
R_DATA = int(round(np.sqrt(np.nanmedian(df['catchment_area_km2']) / np.pi) * 1000 / 50) * 50) or 500
print(f'Stations: {N} | operational: {int(gdf_m.operational.sum())} | Phase 2: {int((~gdf_m.operational).sum())} '
      f'| unreliable positions: {int((~gdf_m.pos_ok).sum())}')
print(f'Dataset catchment radius inferred from catchment_area_km2: {R_DATA} m')
display(gdf_m.groupby(['status', 'line']).size().rename('stations').to_frame())

## 4️⃣ Network Geometry: Station Spacing, Buffers & Overlap
**Why it matters:** wide spacing reduces walk access; heavy buffer overlap means stations compete for the same riders. Interchange pairs closer than 50 m are treated as co-located.

In [ ]:
Dn = D.copy()
np.fill_diagonal(Dn, np.inf)
Dn[Dn < 50] = np.inf                       # ignore co-located interchange duplicates
gdf_m['nn_dist_m'] = Dn.min(axis=1)

same = np.full(N, np.nan)
for ln, idx in gdf_m.groupby('line').indices.items():
    if len(idx) > 1:
        same[idx] = Dn[np.ix_(idx, idx)].min(axis=1)
gdf_m['nn_same_line_m'] = same

spacing = (gdf_m.groupby(['status', 'line'])['nn_same_line_m']
           .agg(stations='size', median_m='median', mean_m='mean', min_m='min', max_m='max').round(0))
display(spacing)

RADII = sorted({400, R_DATA, 800, 1200})
def union_stats(pts, r):
    b = pts.geometry.buffer(r)
    return unary_union(list(b)).area / 1e6, b.area.sum() / 1e6

rows = []
for scope, sub in [('Operational (Phase 1)', gdf_m[gdf_m.operational]), ('Full network (Phase 1 + 2)', gdf_m)]:
    for r in RADII:
        u, s = union_stats(sub, r)
        rows.append({'network': scope, 'radius_m': r, 'stations': len(sub), 'sum_of_buffers_km2': s,
                     'union_km2': u, 'overlap_pct': 100 * (1 - u / s)})
buf_tbl = pd.DataFrame(rows).round(2)
display(buf_tbl)

fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
sns.histplot(gdf_m['nn_same_line_m'].dropna(), bins=25, kde=True, color='#1f5fbf', ax=ax[0])
ax[0].axvline(gdf_m['nn_same_line_m'].median(), color='k', ls='--', label='median')
ax[0].set(title='Nearest-station distance (same line)', xlabel='metres'); ax[0].legend()
sns.boxplot(data=gdf_m, x='line', y='nn_same_line_m', hue='line', palette={l: line_color(l) for l in gdf_m.line.unique()},
            legend=False, ax=ax[1]); ax[1].set(title='Station spacing by line', xlabel='', ylabel='metres')
ax[1].tick_params(axis='x', rotation=20)
sns.barplot(data=buf_tbl, x='radius_m', y='overlap_pct', hue='network', ax=ax[2])
ax[2].set(title='Buffer overlap vs. radius', xlabel='Buffer radius (m)', ylabel='Overlap (%)')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/01_network_geometry.png'); plt.show()

## 5️⃣ Coverage Surface & Phase 2 Coverage Gain
A **250 m raster grid** is laid over the study area (convex hull of all stations + 1.5 km). Each cell gets its straight-line distance to the nearest station, once for the *operational* network and once for the *full* network. The difference is the **coverage gain from Phase 2**.

In [ ]:
hull = unary_union(list(gdf_m.geometry)).convex_hull.buffer(1500)
minx, miny, maxx, maxy = hull.bounds
CELL = 250
xs = np.arange(minx, maxx, CELL) + CELL / 2
ys = np.arange(miny, maxy, CELL) + CELL / 2
gx, gy = np.meshgrid(xs, ys)
inside = shapely.contains_xy(hull, gx.ravel(), gy.ravel()).reshape(gx.shape)
grid_pts = np.c_[gx.ravel(), gy.ravel()]

def dist_surface(mask):
    d, _ = cKDTree(XY[mask]).query(grid_pts)
    return np.where(inside, d.reshape(gx.shape), np.nan)

d_op  = dist_surface(gdf_m.operational.values)
d_all = dist_surface(np.ones(N, dtype=bool))

THRESH = [400, 800, 1000, 1500]
cov = pd.DataFrame({
    'threshold_m': THRESH,
    'operational_%': [100 * np.nanmean(d_op[inside] <= t) for t in THRESH],
    'full_network_%': [100 * np.nanmean(d_all[inside] <= t) for t in THRESH]})
cov['gain_pp'] = cov['full_network_%'] - cov['operational_%']
cov = cov.round(1)
print(f'Study area: {inside.sum() * CELL**2 / 1e6:,.0f} km2 (hull of all stations + 1.5 km)')
display(cov)

bins = [0, 400, 800, 1200, 1600, np.inf]
cls_cols = ['#1a9850', '#91cf60', '#fee08b', '#fc8d59', '#d73027']
cls_lbl = ['<= 400 m', '400-800 m', '800-1200 m', '1200-1600 m', '> 1600 m']
from matplotlib.colors import ListedColormap
cmap = ListedColormap(cls_cols)

fig, axes = plt.subplots(1, 2, figsize=(17, 8), sharex=True, sharey=True)
for ax, surf, ttl in [(axes[0], d_op, 'Operational network (Phase 1)'), (axes[1], d_all, 'Full network (Phase 1 + Phase 2)')]:
    cls = np.where(np.isnan(surf), np.nan, np.digitize(surf, bins[1:-1]))
    ax.pcolormesh(xs, ys, cls, cmap=cmap, vmin=0, vmax=4, shading='nearest', alpha=.85)
    (align_m[~align_m['is_planned']] if surf is d_op else align_m).plot(ax=ax, color='#333', lw=1, alpha=.6)
    sub = gdf_m[gdf_m.operational] if surf is d_op else gdf_m
    ax.scatter(sub.geometry.x, sub.geometry.y, s=14, c=sub['line_color'], edgecolor='k', lw=.4, zorder=5)
    ax.set_title(ttl); ax.set_aspect('equal')
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, p: f'{v/1000:.0f}'))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: f'{v/1000:.0f}'))
    ax.set_xlabel('Easting, UTM 44N (km)')
axes[0].set_ylabel('Northing, UTM 44N (km)'); axes[1].set_ylabel('')
fig.legend(handles=[mpatches.Patch(color=c, label=l) for c, l in zip(cls_cols, cls_lbl)],
           title='Distance to nearest station', loc='lower center', ncol=5, frameon=True)
plt.suptitle('Straight-line distance to the nearest metro station', fontsize=14, fontweight='bold')
plt.tight_layout(rect=(0, .08, 1, 1)); plt.savefig(f'{FIG_DIR}/02_coverage_surface.png'); plt.show()

## 6️⃣ Overlap-Adjusted Catchment Population & Hansen Gravity Accessibility
**Overlap-adjusted population.** Neighbouring buffers share area, so summing catchment populations double-counts people. Each point inside a station's buffer is shared equally among all stations covering it: `unique_share = mean(1/k)` where *k* is the number of stations covering that point. Adjusted population = `population × unique_share` (assumes uniform density inside each buffer).

**Hansen accessibility.** For station *i*: &nbsp; $A_i = \sum_{j \neq i} O_j \, e^{-\beta d_{ij}}$ &nbsp; with opportunities $O_j$ (activity POIs or population in catchment *j*) and $\beta = \ln 2 / h$, where *h* is the distance at which attractiveness halves (base case 3 km). Higher = more opportunities reachable nearby.

In [ ]:
tree_all = cKDTree(XY)
step = 25
loc = np.arange(-R_DATA + step / 2, R_DATA, step)
lx, ly = np.meshgrid(loc, loc)
keep = (lx**2 + ly**2) <= R_DATA**2
local = np.c_[lx[keep], ly[keep]]

share, excl = np.zeros(N), np.zeros(N)
for i in range(N):
    k = tree_all.query_ball_point(XY[i] + local, R_DATA, return_length=True)
    share[i] = np.mean(1.0 / k)
    excl[i]  = np.mean(k == 1)
gdf_m['unique_share']    = share
gdf_m['exclusive_share'] = excl
gdf_m['pop_adj']         = gdf_m['population'] * gdf_m['unique_share']

def hansen(values, half_life_m):
    beta = np.log(2) / half_life_m
    K = np.exp(-beta * D)
    np.fill_diagonal(K, 0)
    return K @ values

HALF_LIFE = 3000
gdf_m['hansen_activity'] = hansen(gdf_m['poi_total'].values, HALF_LIFE)
gdf_m['hansen_pop']      = hansen(gdf_m['population'].values, HALF_LIFE)

# sensitivity of the ranking to the decay parameter
sens = pd.DataFrame({f'{h/1000:g} km': hansen(gdf_m['poi_total'].values, h) for h in [1500, 3000, 6000]})
print('Spearman rank agreement of Hansen(activity) across decay half-lives:')
display(sens.corr(method='spearman').round(3))

tot_raw, tot_adj = gdf_m['population'].sum(), gdf_m['pop_adj'].sum()
print(f'Sum of catchment populations: {tot_raw:,.0f} | overlap-adjusted: {tot_adj:,.0f} '
      f'({100 * (1 - tot_adj / tot_raw):.1f}% double-counting removed)')
display(gdf_m.groupby('status')[['population', 'pop_adj', 'population_2030']].sum().round(0))
display(gdf_m.nlargest(8, 'hansen_activity')[['name', 'line', 'status', 'hansen_activity', 'hansen_pop']].round(0))

## 7️⃣ Urban Catchment Profile (Urban-Form Indicators)
Derived indicators used later in the TOD index:
* **Land-use mix (entropy)** – $H = -\sum p_k \ln p_k / \ln K$ over the seven POI categories (0 = single use, 1 = perfectly mixed).
* **POI richness** – number of POI categories present.
* **Walk efficiency** – street-network catchment area ÷ circular catchment area (low = barriers, severance, cul-de-sacs).
* **Growth 2030** – projected population growth in the catchment (used for prioritisation, *not* in the index).

In [ ]:
POI_COLS = [c for c in ['poi_education', 'poi_healthcare', 'poi_retail', 'poi_office',
                        'poi_government', 'poi_transport', 'poi_leisure'] if c in gdf_m]
Pm = gdf_m[POI_COLS].clip(lower=0).values
tot = Pm.sum(axis=1, keepdims=True)
with np.errstate(all='ignore'):
    p = np.where(tot > 0, Pm / tot, 0.0)
    ent = -np.where(p > 0, p * np.log(p), 0.0).sum(axis=1) / np.log(len(POI_COLS))
gdf_m['poi_mix_entropy'] = ent
gdf_m['poi_richness']    = (Pm > 0).sum(axis=1)

walk = gdf_m['network_catchment_km2'] / gdf_m['catchment_area_km2']
gdf_m['walk_efficiency'] = walk.replace([np.inf, -np.inf], np.nan).fillna(walk.median())
gdf_m['growth_2030']     = (gdf_m['population_2030'] / gdf_m['population'].replace(0, np.nan) - 1).fillna(0)
gdf_m['activity_per_1000_residents'] = gdf_m['poi_total'] / gdf_m['population'].replace(0, np.nan) * 1000

if 'pnr' in gdf_m:
    print('Sanity check - corr(walk_efficiency, dataset `pnr`):',
          round(gdf_m[['walk_efficiency', 'pnr']].corr().iloc[0, 1], 3))

prof_cols = ['population_density', 'built_volume_density', 'intersection_density', 'street_density', 'circuity',
             'walk_efficiency', 'poi_total', 'poi_mix_entropy', 'bus_stops_500m', 'dist_cbd_km', 'growth_2030']
prof_cols = [c for c in prof_cols if c in gdf_m]
display(gdf_m.groupby('status')[prof_cols].median().round(2).T)

fig, ax = plt.subplots(2, 3, figsize=(18, 9))
sns.boxplot(data=gdf_m, x='status', y='walk_efficiency', hue='status', legend=False, ax=ax[0, 0]); ax[0, 0].set(title='Walk efficiency (network / circle)', xlabel='')
sns.histplot(data=gdf_m, x='poi_mix_entropy', hue='status', bins=20, kde=True, ax=ax[0, 1]); ax[0, 1].set(title='Land-use mix entropy')
sns.scatterplot(data=gdf_m, x='intersection_density', y='walk_efficiency', hue='status', size='population_density', sizes=(20, 200), alpha=.7, ax=ax[0, 2])
ax[0, 2].set(title='Street connectivity vs. walk efficiency')
mix = gdf_m.groupby('line')[POI_COLS].sum(); mix = mix.div(mix.sum(axis=1), axis=0)
mix.rename(columns=lambda c: c.replace('poi_', '')).plot.bar(stacked=True, colormap='tab10', ax=ax[1, 0])
ax[1, 0].set(title='POI composition by line', xlabel='', ylabel='share'); ax[1, 0].tick_params(axis='x', rotation=20); ax[1, 0].legend(fontsize=8, ncol=2)
sns.violinplot(data=gdf_m, x='line', y='population_density', hue='line', legend=False, inner='quartile',
               palette={l: line_color(l) for l in gdf_m.line.unique()}, ax=ax[1, 1])
ax[1, 1].set(title='Residential density by line', xlabel=''); ax[1, 1].tick_params(axis='x', rotation=20)
sns.scatterplot(data=gdf_m, x='dist_cbd_km', y='population_density', hue='status', alpha=.75, ax=ax[1, 2])
ax[1, 2].set(title='Density decay with distance from CBD')
plt.suptitle('Urban catchment profile of Chennai Metro stations', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/03_catchment_profile.png'); plt.show()

## 8️⃣ TOD Potential Index (5D Framework)
| Dimension | Indicators (direction) | Meaning |
|---|---|---|
| **Density** | population density (+), built-volume density (+), total POIs (+) | Enough people and activity within walking distance |
| **Diversity** | land-use entropy (+), POI richness (+) | Mixed use → all-day demand, shorter trips |
| **Design** | intersection density (+), street density (+), walk efficiency (+), circuity (−) | Pedestrian-friendly, permeable street network |
| **Destination accessibility** | Hansen activity accessibility (+), distance to CBD (−), distance to nearest centre (−) | Regional reach of jobs and services by metro |
| **Distance to transit / multimodality** | feeder bus stops (+), metro closeness / degree / betweenness (+), competing rail within 1 km (+), interchange (+) | Quality of connections to and within the network |

**Scoring:** each indicator is converted to a **percentile rank (0-100)** across all 134 stations (direction-corrected, so outliers cannot dominate and skewed variables need no transform); dimension score = mean of its indicators; **TOD index = weighted mean of the five dimension scores.**

**Weight schemes (robustness):** (A) *Equal* (base case) · (B) *Expert* – density-heavy · (C) *CRITIC* – objective weights from dispersion and redundancy of the dimensions.

In [ ]:
DIMS = {
    'Density':      [('population_density', +1), ('built_volume_density', +1), ('poi_total', +1)],
    'Diversity':    [('poi_mix_entropy', +1), ('poi_richness', +1)],
    'Design':       [('intersection_density', +1), ('street_density', +1), ('walk_efficiency', +1), ('circuity', -1)],
    'Destination':  [('hansen_activity', +1), ('dist_cbd_km', -1), ('dist_nearest_centre_km', -1)],
    'Transit_link': [('bus_stops_500m', +1), ('metro_closeness', +1), ('metro_degree', +1),
                     ('metro_betweenness', +1), ('competing_rail_1km', +1), ('is_interchange', +1)],
}
def pct_score(s, sign):
    r = s.rank(pct=True) * 100
    return r if sign > 0 else (100 - r + 100 / len(s))

ind_scores, used = {}, {}
dim_scores = pd.DataFrame(index=gdf_m.index)
for dim, items in DIMS.items():
    cols = []
    for col, sign in items:
        if col in gdf_m and gdf_m[col].nunique(dropna=True) > 1:
            ind_scores[col] = pct_score(gdf_m[col].fillna(gdf_m[col].median()), sign)
            cols.append(col)
    used[dim] = cols
    dim_scores[dim] = pd.concat([ind_scores[c] for c in cols], axis=1).mean(axis=1)
print('Indicators actually used per dimension:')
for k, v in used.items():
    print(f'  {k:13s}', v)

K = dim_scores.shape[1]
w_equal  = pd.Series(1 / K, index=dim_scores.columns)
w_expert = pd.Series({'Density': .25, 'Diversity': .15, 'Design': .20, 'Destination': .20, 'Transit_link': .20}).reindex(dim_scores.columns).fillna(0)
w_expert = w_expert / w_expert.sum()
Z = (dim_scores - dim_scores.min()) / (dim_scores.max() - dim_scores.min())
critic = Z.std(ddof=1) * (1 - Z.corr()).sum()
w_critic = critic / critic.sum()
weights = pd.DataFrame({'A_Equal': w_equal, 'B_Expert': w_expert, 'C_CRITIC': w_critic}).round(3)
display(weights)

schemes = pd.DataFrame({'A_Equal': dim_scores @ w_equal, 'B_Expert': dim_scores @ w_expert, 'C_CRITIC': dim_scores @ w_critic})
print('Spearman agreement between weight schemes (rank correlation):')
display(schemes.corr(method='spearman').round(3))

for c in dim_scores.columns:
    gdf_m[c] = dim_scores[c]
gdf_m['TOD_index'] = schemes['A_Equal']
gdf_m['TOD_rank']  = gdf_m['TOD_index'].rank(ascending=False, method='first').astype(int)
gdf_m['TOD_tier']  = pd.qcut(gdf_m['TOD_rank'], 4, labels=['Tier 1 - Prime', 'Tier 2 - High', 'Tier 3 - Emerging', 'Tier 4 - Limited']).astype(str)
TIER_COL = {'Tier 1 - Prime': '#1a9850', 'Tier 2 - High': '#91cf60', 'Tier 3 - Emerging': '#fdae61', 'Tier 4 - Limited': '#d73027'}
DIM_NAMES = list(dim_scores.columns)

print('\nTop 10 stations by TOD index:')
display(gdf_m.nsmallest(10, 'TOD_rank')[['TOD_rank', 'name', 'line', 'status', 'TOD_index'] + DIM_NAMES].round(1).reset_index(drop=True))
print('Bottom 5:')
display(gdf_m.nlargest(5, 'TOD_rank')[['TOD_rank', 'name', 'line', 'status', 'TOD_index']].round(1).reset_index(drop=True))

### 8.1 Robustness: Monte-Carlo Weight Sensitivity
Weights are subjective, so 2,000 random weight vectors are drawn from a **Dirichlet distribution centred on the equal weights** and the ranking is recomputed each time. Narrow rank intervals and a high *P(top 20%)* mean the station's position does not depend on the weight choice.

In [ ]:
rng = np.random.default_rng(RNG_SEED)
W = rng.dirichlet(w_equal.values * 30, size=2000)              # (2000, K)
S = dim_scores.values @ W.T                                    # (N, 2000)
ranks = (-S).argsort(axis=0).argsort(axis=0) + 1
top_cut = math.ceil(0.20 * N)
gdf_m['rank_p05']  = np.percentile(ranks, 5, axis=1)
gdf_m['rank_p50']  = np.percentile(ranks, 50, axis=1)
gdf_m['rank_p95']  = np.percentile(ranks, 95, axis=1)
gdf_m['p_top20']   = (ranks <= top_cut).mean(axis=1)
gdf_m['rank_span'] = gdf_m['rank_p95'] - gdf_m['rank_p05']

print(f'Median 5-95% rank interval width: {gdf_m.rank_span.median():.0f} places (out of {N})')
stable = gdf_m[(gdf_m.p_top20 >= .9)]
print(f'Stations in the top 20% in >= 90% of weight draws: {len(stable)}')
display(gdf_m.nsmallest(10, 'TOD_rank')[['TOD_rank', 'name', 'rank_p05', 'rank_p50', 'rank_p95', 'p_top20']].round(2).reset_index(drop=True))

## 9️⃣ Node–Place Model & Station Typology
**Node–Place (Bertolini).** *Node* = how well the station is connected and how much it can reach (mean of *Transit_link* and *Destination*). *Place* = how intense and walkable the surroundings are (mean of *Density*, *Diversity*, *Design*). Splitting both axes at the network median gives four policy-relevant classes.

**Typology.** K-Means on the five standardised dimension scores; *k* chosen by silhouette (2-6).

In [ ]:
gdf_m['Node']  = gdf_m[['Transit_link', 'Destination']].mean(axis=1)
gdf_m['Place'] = gdf_m[['Density', 'Diversity', 'Design']].mean(axis=1)
nm, pm = gdf_m['Node'].median(), gdf_m['Place'].median()

def node_place_class(r):
    hn, hp = r['Node'] >= nm, r['Place'] >= pm
    if hn and hp:      return 'Mature TOD node'
    if hn and not hp:  return 'Under-built node (intensify)'
    if not hn and hp:  return 'Under-served place (improve access)'
    return 'Low-intensity periphery (land-bank)'
gdf_m['NodePlace'] = gdf_m.apply(node_place_class, axis=1)
NP_COL = {'Mature TOD node': '#1a9850', 'Under-built node (intensify)': '#e6850e',
          'Under-served place (improve access)': '#3b78c3', 'Low-intensity periphery (land-bank)': '#9aa0a6'}
display(pd.crosstab(gdf_m['NodePlace'], gdf_m['status'], margins=True))

Xs = StandardScaler().fit_transform(gdf_m[DIM_NAMES])
sil = {k: silhouette_score(Xs, KMeans(k, n_init=20, random_state=RNG_SEED).fit_predict(Xs)) for k in range(2, 7)}
K_BEST = max(sil, key=sil.get)
print('Silhouette by k:', {k: round(v, 3) for k, v in sil.items()}, '-> k =', K_BEST)
km = KMeans(K_BEST, n_init=50, random_state=RNG_SEED).fit(Xs)
gdf_m['cluster'] = km.labels_
cz = pd.DataFrame(km.cluster_centers_, columns=DIM_NAMES)

def name_cluster(row):
    top = row[row > 0.35].sort_values(ascending=False).index.tolist()[:2]
    if not top:
        return 'Low-intensity'
    return ' + '.join(t.replace('_', ' ') for t in top) + '-led'
cl_names = {i: f'T{i+1}: {name_cluster(cz.loc[i])}' for i in cz.index}
gdf_m['typology'] = gdf_m['cluster'].map(cl_names)

prof = gdf_m.groupby('typology')[DIM_NAMES + ['TOD_index']].mean().round(1)
prof['stations'] = gdf_m.groupby('typology').size()
prof['phase2_share_%'] = (gdf_m.groupby('typology')['operational'].apply(lambda s: 100 * (1 - s.mean()))).round(0)
display(prof)
plt.figure(figsize=(9, 3 + .5 * K_BEST))
sns.heatmap(cz.rename(index=cl_names), annot=True, fmt='.2f', cmap='RdBu_r', center=0, linewidths=.5)
plt.title('Typology profile (standardised dimension means)'); plt.tight_layout()
plt.savefig(f'{FIG_DIR}/04_typology_heatmap.png'); plt.show()

## 🔟 Spatial Statistics: Are TOD Scores Spatially Clustered?
* **Global Moran's I** (k = 5 nearest-neighbour weights, row-standardised, 999 permutations): $I = \frac{n}{S_0}\frac{\sum_i\sum_j w_{ij} z_i z_j}{\sum_i z_i^2}$. *I > 0* → similar values cluster; *I ≈ 0* → random.
* **Getis-Ord Gi\*** (k = 5 neighbours + self): z-scores identify **hot spots** (clusters of high TOD scores) and **cold spots**.

*With only 134 points and 5 neighbours the local tests are exploratory; no multiple-testing correction is applied.*

In [ ]:
def knn_binary(Dm, k=5):
    Dm = Dm.copy(); np.fill_diagonal(Dm, np.inf)
    idx = np.argsort(Dm, axis=1)[:, :k]
    Wb = np.zeros_like(Dm)
    for i in range(len(Dm)):
        Wb[i, idx[i]] = 1
    return Wb

def morans_i(x, Wr, perms=999, seed=RNG_SEED):
    z = x - x.mean()
    I = (z @ Wr @ z) / (z @ z)
    rg = np.random.default_rng(seed)
    sims = np.array([(zp @ Wr @ zp) / (zp @ zp) for zp in (rg.permutation(z) for _ in range(perms))])
    ez = sims.mean()
    p = (1 + np.sum(np.abs(sims - ez) >= abs(I - ez))) / (perms + 1)
    return I, (I - ez) / sims.std(), p

def gi_star(x, Wb):
    n = len(x); Ws = Wb + np.eye(n)
    xbar, S = x.mean(), np.sqrt((x ** 2).mean() - x.mean() ** 2)
    sw, sw2 = Ws.sum(1), (Ws ** 2).sum(1)
    return (Ws @ x - xbar * sw) / (S * np.sqrt((n * sw2 - sw ** 2) / (n - 1)))

Wb = knn_binary(D, 5)
Wr = Wb / Wb.sum(axis=1, keepdims=True)

mi_rows = []
for v in ['TOD_index', 'Node', 'Place'] + DIM_NAMES + ['population_density', 'hansen_activity']:
    I, z, p = morans_i(gdf_m[v].values.astype(float), Wr)
    mi_rows.append({'variable': v, "Moran's I": I, 'z': z, 'p (perm.)': p, 'pattern': 'clustered' if (p < .05 and I > 0) else ('dispersed' if (p < .05 and I < 0) else 'not significant')})
moran_tbl = pd.DataFrame(mi_rows).round(3)
display(moran_tbl)

gdf_m['gi_z'] = gi_star(gdf_m['TOD_index'].values.astype(float), Wb)
def hot_class(z):
    if z >= 2.58: return 'Hot spot 99%'
    if z >= 1.96: return 'Hot spot 95%'
    if z >= 1.645: return 'Hot spot 90%'
    if z <= -2.58: return 'Cold spot 99%'
    if z <= -1.96: return 'Cold spot 95%'
    if z <= -1.645: return 'Cold spot 90%'
    return 'Not significant'
gdf_m['hotspot'] = gdf_m['gi_z'].map(hot_class)
print(gdf_m['hotspot'].value_counts().to_string())

z = (gdf_m['TOD_index'] - gdf_m['TOD_index'].mean()).values
lag = Wr @ z
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
ax[0].scatter(z, lag, c=[TIER_COL[t] for t in gdf_m['TOD_tier']], edgecolor='k', lw=.4, s=60)
mfit = np.polyfit(z, lag, 1); xx = np.linspace(z.min(), z.max(), 10)
ax[0].plot(xx, np.polyval(mfit, xx), 'k--', label=f'slope = {mfit[0]:.2f} (= Moran I)')
ax[0].axhline(0, c='grey', lw=.8); ax[0].axvline(0, c='grey', lw=.8)
ax[0].set(title="Moran scatter plot - TOD index", xlabel='TOD index (centred)', ylabel='Spatial lag (mean of 5 neighbours)'); ax[0].legend()
for (tx, ty, t) in [(.97, .97, 'High-High'), (.03, .97, 'Low-High'), (.03, .03, 'Low-Low'), (.97, .03, 'High-Low')]:
    ax[0].text(tx, ty, t, transform=ax[0].transAxes, ha='right' if tx > .5 else 'left', va='top' if ty > .5 else 'bottom', color='grey', fontsize=10)
hc = {'Hot spot 99%': '#b2182b', 'Hot spot 95%': '#ef8a62', 'Hot spot 90%': '#fddbc7', 'Not significant': '#d9d9d9',
      'Cold spot 90%': '#d1e5f0', 'Cold spot 95%': '#67a9cf', 'Cold spot 99%': '#2166ac'}
align_m.plot(ax=ax[1], color='#999', lw=1)
for k, c in hc.items():
    s = gdf_m[gdf_m.hotspot == k]
    if len(s): ax[1].scatter(s.geometry.x, s.geometry.y, c=c, s=70, edgecolor='k', lw=.4, label=f'{k} ({len(s)})', zorder=5)
ax[1].set_aspect('equal'); ax[1].set_title('Getis-Ord Gi* hot / cold spots of TOD potential'); ax[1].legend(fontsize=8, loc='lower right')
ax[1].xaxis.set_major_formatter(FuncFormatter(lambda v, p: f'{v/1000:.0f}')); ax[1].yaxis.set_major_formatter(FuncFormatter(lambda v, p: f'{v/1000:.0f}'))
ax[1].set_xlabel('Easting (km)'); ax[1].set_ylabel('Northing (km)')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/05_spatial_statistics.png'); plt.show()

## 1️⃣1️⃣ Validation Against Observed Ridership (Operational Stations)
Station names in `cmrl_station_summary.csv` come from the CMRL API, so they are matched to the station directory by **normalised-name exact match, then fuzzy match** (difflib ≥ 0.82). The match table is printed so it can be **checked by hand**; fix mismatches through `MANUAL_NAME_MAP`. Only `is_mature` stations are used (young stations are still ramping up).

Checks: (1) Spearman correlation of TOD index and each dimension with weekday boardings; (2) a small **Ridge regression** on the five dimension scores with **leave-one-out CV** (n is small, so a simple, regularised model is used); (3) **over/under-performers** = station residuals.

In [ ]:
MANUAL_NAME_MAP = {}     # e.g. {'AIRPORT': 'Chennai International Airport'}
HAVE_RIDERSHIP = False
val = None

def norm_name(s):
    s = re.sub(r'\(.*?\)', ' ', str(s).lower())
    s = re.sub(r'\b(metro|station|stn)\b', ' ', s)
    return re.sub(r'[^a-z0-9]', '', s)

if summary is not None and {'boardings_weekday'} <= set(summary.columns):
    op_df = gdf_m[gdf_m.operational]
    keys = {}
    for i, n in op_df['name'].items():
        keys.setdefault(norm_name(n), i)
    match_rows = []
    for _, r in summary.iterrows():
        cands = [c for c in [r.get('canonical_station_name'), r.get('cmrl_name')] if pd.notna(c)]
        hit, how = None, None
        for c in cands:
            k = norm_name(MANUAL_NAME_MAP.get(c, c))
            if k in keys: hit, how = keys[k], 'exact'; break
        if hit is None:
            for c in cands:
                m = difflib.get_close_matches(norm_name(MANUAL_NAME_MAP.get(c, c)), list(keys), n=1, cutoff=0.82)
                if m: hit, how = keys[m[0]], 'fuzzy'; break
        match_rows.append({'summary_name': cands[0] if cands else None, 'matched_station': None if hit is None else gdf_m.at[hit, 'name'], 'method': how or 'UNMATCHED', 'idx': hit})
    mt = pd.DataFrame(match_rows)
    print('Match methods:', mt['method'].value_counts().to_dict())
    if (mt.method == 'UNMATCHED').any():
        print('Unmatched summary rows (add to MANUAL_NAME_MAP):', mt.loc[mt.method == 'UNMATCHED', 'summary_name'].tolist())
    if (mt.method == 'fuzzy').any():
        display(mt[mt.method == 'fuzzy'].drop(columns='idx'))
    mt = mt.dropna(subset=['idx']).drop_duplicates('idx')
    sm = summary.loc[mt.index].copy()
    sm['idx'] = mt['idx'].astype(int).values
    val = gdf_m.loc[sm['idx']].copy()
    for c in ['boardings_weekday', 'boardings_weekend', 'weekend_ratio', 'is_mature']:
        if c in sm: val[c] = sm[c].values
    val['is_mature'] = val['is_mature'].astype(str).str.lower().isin(['true', '1', 'yes']) if 'is_mature' in val else True
    val = val[val['is_mature']].copy()
    HAVE_RIDERSHIP = len(val) >= 8
    print(f'Stations with usable ridership + TOD scores (mature): {len(val)}')
else:
    print('Station summary not available - validation section skipped.')

In [ ]:
if HAVE_RIDERSHIP:
    val['log_board'] = np.log1p(val['boardings_weekday'])
    rho, pv = stats.spearmanr(val['TOD_index'], val['boardings_weekday'])
    print(f'Spearman rho (TOD index vs weekday boardings) = {rho:.2f}  (p = {pv:.3f}, n = {len(val)})')
    dim_rho = pd.DataFrame({d: tuple(stats.spearmanr(val[d], val['boardings_weekday'])) for d in DIM_NAMES + ['Node', 'Place']},
                           index=['rho', 'p']).T.round(3)
    display(dim_rho)

    feat_cols = [c for c in ['population_density', 'population', 'built_volume_density', 'poi_total', 'poi_office', 'poi_education',
                             'poi_retail', 'bus_stops_500m', 'intersection_density', 'walk_efficiency', 'hansen_activity',
                             'metro_betweenness', 'metro_closeness', 'dist_cbd_km', 'poi_mix_entropy', 'is_interchange'] if c in val]
    feat_rho = pd.Series({c: stats.spearmanr(val[c], val['boardings_weekday'])[0] for c in feat_cols}).sort_values()

    # ridge on 5 dimension scores, leave-one-out CV
    Xv, yv = val[DIM_NAMES].values, val['log_board'].values
    ridge = RidgeCV(alphas=np.logspace(-2, 4, 40))
    loo = cross_val_predict(ridge, Xv, yv, cv=LeaveOneOut())
    r2_loo, mae_loo = r2_score(yv, loo), mean_absolute_error(np.expm1(yv), np.expm1(loo))
    ridge.fit(Xv, yv)
    print(f'Ridge (5 dimension scores) | LOO R2 = {r2_loo:.2f} | LOO MAE = {mae_loo:,.0f} boardings/weekday | alpha = {ridge.alpha_:.2f}')
    print('Standardised importance (coef x std):', {d: round(float(v), 3) for d, v in zip(DIM_NAMES, ridge.coef_ * Xv.std(0))})

    val['pred_loo'] = np.expm1(loo)
    val['perf_gap_%'] = 100 * (val['boardings_weekday'] / val['pred_loo'] - 1)
    val['performance'] = pd.cut(val['perf_gap_%'], [-np.inf, -25, 25, np.inf], labels=['Under-performing', 'As expected', 'Over-performing']).astype(str)
    print(val['performance'].value_counts().to_string())

    fig, ax = plt.subplots(1, 3, figsize=(20, 6))
    sc = ax[0].scatter(val['TOD_index'], val['boardings_weekday'], c=val['Node'], cmap='viridis', s=80, edgecolor='k', lw=.4)
    mf = np.polyfit(val['TOD_index'], val['boardings_weekday'], 1); xx = np.linspace(val['TOD_index'].min(), val['TOD_index'].max(), 10)
    ax[0].plot(xx, np.polyval(mf, xx), 'r--'); plt.colorbar(sc, ax=ax[0], label='Node score')
    ax[0].set(title=f'TOD index vs weekday boardings (rho = {rho:.2f})', xlabel='TOD index', ylabel='Avg weekday boardings')
    for _, r in val.nlargest(4, 'boardings_weekday').iterrows():
        ax[0].annotate(r['name'], (r['TOD_index'], r['boardings_weekday']), fontsize=8, xytext=(4, 4), textcoords='offset points')
    ax[1].barh(feat_rho.index, feat_rho.values, color=np.where(feat_rho.values > 0, '#2c7fb8', '#d95f0e'))
    ax[1].axvline(0, c='k', lw=.8); ax[1].set(title='Spearman rho with weekday boardings', xlabel='rho')
    lim = [0, max(val['boardings_weekday'].max(), val['pred_loo'].max()) * 1.05]
    ax[2].scatter(val['pred_loo'], val['boardings_weekday'], c=val['performance'].map({'Under-performing': '#d73027', 'As expected': '#999', 'Over-performing': '#1a9850'}), s=70, edgecolor='k', lw=.4)
    ax[2].plot(lim, lim, 'k--'); ax[2].set(title=f'Observed vs LOO-predicted (R2 = {r2_loo:.2f})', xlabel='Predicted', ylabel='Observed', xlim=lim, ylim=lim)
    plt.tight_layout(); plt.savefig(f'{FIG_DIR}/06_ridership_validation.png'); plt.show()

    cols = ['name', 'line', 'boardings_weekday', 'pred_loo', 'perf_gap_%', 'TOD_index', 'NodePlace']
    print('Most UNDER-performing vs built-environment expectation (unrealised demand):')
    display(val.nsmallest(5, 'perf_gap_%')[cols].round(1).reset_index(drop=True))
    print('Most OVER-performing:')
    display(val.nlargest(5, 'perf_gap_%')[cols].round(1).reset_index(drop=True))

    if 'weekend_ratio' in val and 'poi_leisure' in val:
        share_leis = (val['poi_leisure'] / val['poi_total'].replace(0, np.nan)).fillna(0)
        r2_, p2_ = stats.spearmanr(share_leis, val['weekend_ratio'])
        r3_, p3_ = stats.spearmanr(val['poi_mix_entropy'], val['weekend_ratio'])
        print(f'Weekend/weekday ratio vs leisure-POI share: rho = {r2_:.2f} (p = {p2_:.3f}); vs land-use entropy: rho = {r3_:.2f} (p = {p3_:.3f})')
else:
    print('Ridership validation skipped (no usable matched stations).')

## 1️⃣2️⃣ Phase 2 TOD Priority List
Phase 2 stations have no ridership history, so priority rests on the TOD index, its **stability** under random weights, the **Node-Place class** (what policy lever is needed) and projected **2030 population growth**. If the validation model above is usable, an *indicative demand-potential* figure is added (Phase 2 stations are partly outside the training range - treat it as a relative signal only). The optional ML predictions in the dataset are compared where available.

In [ ]:
p2 = gdf_m[~gdf_m.operational].copy()
if HAVE_RIDERSHIP:
    p2['demand_potential'] = np.expm1(ridge.predict(p2[DIM_NAMES].values))
    gdf_m.loc[p2.index, 'demand_potential'] = p2['demand_potential']

    # compare with the dataset's own ML predictions if present
    if pred_file is not None:
        pcols = [c for c in pred_file.columns if 'pred' in c.lower() and not any(t in c.lower() for t in ['low', 'upp', 'lo_', 'hi_', 'ci', 'interval'])]
        if pcols and 'station_id' in pred_file.columns:
            cmp_ = p2[['station_id', 'demand_potential']].merge(pred_file[['station_id', pcols[0]]], on='station_id')
            if len(cmp_) > 5:
                r_, p_ = stats.spearmanr(cmp_['demand_potential'], cmp_[pcols[0]])
                print(f"Rank agreement with dataset ML predictions ('{pcols[0]}') on {len(cmp_)} Phase 2 stations: rho = {r_:.2f} (p = {p_:.3f})")

p2['priority_score'] = (0.6 * p2['TOD_index'].rank(pct=True) + 0.2 * p2['growth_2030'].rank(pct=True) + 0.2 * p2['p_top20']) * 100
p2['priority_rank'] = p2['priority_score'].rank(ascending=False, method='first').astype(int)
gdf_m.loc[p2.index, 'priority_score'] = p2['priority_score']
gdf_m.loc[p2.index, 'priority_rank']  = p2['priority_rank']

show = ['priority_rank', 'name', 'line', 'TOD_index', 'TOD_tier', 'NodePlace', 'typology', 'pop_adj', 'growth_2030', 'p_top20']
print('Phase 2 priority score = 60% TOD index percentile + 20% growth-2030 percentile + 20% rank stability (P top-20%)')
display(p2.nsmallest(15, 'priority_rank')[show].round(2).reset_index(drop=True))
print('Phase 2 stations by policy class:')
display(pd.crosstab(p2['line'], p2['NodePlace']))

## 1️⃣3️⃣ Geospatial Maps
### 13.1 Static map: TOD tiers across the network
Marker colour = TOD tier, size = TOD index, circles = operational, triangles = Phase 2, dashed lines = planned / under-construction alignment. A web basemap is added when `contextily` and internet access are available.

In [ ]:
try:
    import contextily as cx
    HAVE_CX = True
except ImportError:
    try:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'contextily'])
        import contextily as cx
        HAVE_CX = True
    except Exception:
        HAVE_CX = False

gw = gdf_m.to_crs(3857)
aw = align_m.to_crs(3857)
fig, ax = plt.subplots(figsize=(13, 13))
aw[~aw['is_planned']].plot(ax=ax, color='#222', lw=2.2, zorder=2)
aw[aw['is_planned']].plot(ax=ax, color='#555', lw=1.6, ls='--', zorder=2)
for mk, flag in [('o', True), ('^', False)]:
    s = gw[gw.operational == flag]
    ax.scatter(s.geometry.x, s.geometry.y, s=20 + s['TOD_index'] * 3.2, marker=mk,
               c=s['TOD_tier'].map(TIER_COL), edgecolor='k', lw=.8, alpha=.92, zorder=5)
for k_, (_, r) in enumerate(gw.nsmallest(6, 'TOD_rank').iterrows()):
    ax.annotate(f"{r['name']}  ({r['TOD_index']:.0f})", (r.geometry.x, r.geometry.y), xytext=(14, 14 + 16 * (k_ % 3) * (1 if k_ % 2 else -1)),
                arrowprops=dict(arrowstyle='-', lw=.6, color='#444'), textcoords='offset points',
                fontsize=8, fontweight='bold', bbox=dict(boxstyle='round,pad=.25', fc='white', alpha=.85, lw=.4), zorder=6)
if HAVE_CX:
    try:
        cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, zoom=11, attribution_size=6)
    except Exception as e:
        print('Basemap unavailable (offline?):', str(e)[:80])
leg = [mpatches.Patch(color=c, label=t) for t, c in TIER_COL.items()]
leg += [plt.Line2D([], [], marker='o', ls='', mfc='w', mec='k', label='Operational'),
        plt.Line2D([], [], marker='^', ls='', mfc='w', mec='k', label='Phase 2'),
        plt.Line2D([], [], color='#222', lw=2, label='Operational alignment'),
        plt.Line2D([], [], color='#555', lw=1.6, ls='--', label='Planned / under construction')]
ax.legend(handles=leg, loc='lower left', frameon=True, framealpha=.95, title='TOD potential')
ax.set_title('Chennai Metro - Transit-Oriented Development Potential by Station', fontsize=15, fontweight='bold')
ax.set_axis_off(); plt.tight_layout(); plt.savefig(f'{FIG_DIR}/07_tod_map.png'); plt.show()

### 13.2 Interactive web map (Folium)
Toggle layers (operational / Phase 2 stations, catchment circles, TOD heat layer). Click a station for its scores. The map is also saved as a stand-alone HTML file in the outputs folder.

In [ ]:
c_lat, c_lon = gdf_m['latitude'].mean(), gdf_m['longitude'].mean()
m = folium.Map(location=[c_lat, c_lon], zoom_start=11, tiles='CartoDB positron', control_scale=True)

fg_al = folium.FeatureGroup('Alignments', show=True)
for _, r in alignments.iterrows():
    planned = pd.notna(r.get('construction')) if 'construction' in alignments else False
    parts = [r.geometry] if r.geometry.geom_type == 'LineString' else (list(r.geometry.geoms) if r.geometry.geom_type == 'MultiLineString' else [])
    for part in parts:
        folium.PolyLine([(y, x) for x, y in part.coords], color='#555' if planned else '#111', weight=2 if planned else 3,
                        dash_array='6 6' if planned else None, opacity=.7).add_to(fg_al)
fg_al.add_to(m)

def popup_html(r):
    return (f"<b>{r['name']}</b><br>{r['line']} | {r['status']} | {r.get('layout', '')}<hr style='margin:3px'>"
            f"<b>TOD index:</b> {r['TOD_index']:.1f} (rank {int(r['TOD_rank'])}/{N}, {r['TOD_tier']})<br>"
            f"<b>Node / Place:</b> {r['Node']:.0f} / {r['Place']:.0f} - {r['NodePlace']}<br>"
            f"<b>Typology:</b> {r['typology']}<br>"
            f"Density {r['Density']:.0f} | Diversity {r['Diversity']:.0f} | Design {r['Design']:.0f} | Destination {r['Destination']:.0f} | Transit {r['Transit_link']:.0f}<br>"
            f"Pop. (overlap-adj.): {r['pop_adj']:,.0f} | bus stops 500 m: {int(r['bus_stops_500m'])}<br>"
            f"Gi* hotspot: {r['hotspot']}")

fg_op, fg_p2, fg_buf = folium.FeatureGroup('Operational stations', show=True), folium.FeatureGroup('Phase 2 stations', show=True), folium.FeatureGroup(f'{R_DATA} m catchments', show=False)
for _, r in gdf_m.iterrows():
    grp = fg_op if r['operational'] else fg_p2
    folium.CircleMarker([r['latitude'], r['longitude']], radius=4 + r['TOD_index'] / 12, color='black', weight=1.2 if r['operational'] else .6,
                        fill=True, fill_color=TIER_COL[r['TOD_tier']], fill_opacity=.9,
                        popup=folium.Popup(popup_html(r), max_width=330), tooltip=f"{r['name']} - TOD {r['TOD_index']:.0f}").add_to(grp)
    folium.Circle([r['latitude'], r['longitude']], radius=R_DATA, color=r['line_color'], weight=1, fill=True, fill_opacity=.05).add_to(fg_buf)
for g in (fg_op, fg_p2, fg_buf):
    g.add_to(m)
HeatMap(gdf_m[['latitude', 'longitude', 'TOD_index']].assign(w=lambda d: d.TOD_index / 100).values[:, [0, 1, 3]].tolist(),
        name='TOD heat layer', radius=28, blur=22, show=False).add_to(m)

legend = ("<div style='position:fixed;bottom:25px;left:25px;z-index:9999;background:white;padding:8px 10px;border:1px solid #888;font-size:12px'>"
          "<b>TOD tier</b><br>" + ''.join(f"<span style='color:{c}'>&#9679;</span> {t}<br>" for t, c in TIER_COL.items()) + "</div>")
m.get_root().html.add_child(folium.Element(legend))
folium.LayerControl(collapsed=False).add_to(m)
MAP_PATH = os.path.join(OUT_DIR, 'chennai_tod_map.html')
m.save(MAP_PATH)
print('Saved:', MAP_PATH)
m

## 1️⃣4️⃣ Interactive Plotly Dashboards

In [ ]:
# (a) Node-Place scatter
fig = px.scatter(gdf_m, x='Place', y='Node', color='NodePlace', color_discrete_map=NP_COL, symbol='status', size='TOD_index', size_max=18,
                 hover_name='name', hover_data={'line': True, 'TOD_index': ':.1f', 'TOD_tier': True, 'typology': True, 'Place': ':.1f', 'Node': ':.1f'},
                 title='Node-Place model: connectivity (Node) vs. urban intensity (Place)', height=620)
fig.add_vline(x=pm, line_dash='dash', line_color='grey'); fig.add_hline(y=nm, line_dash='dash', line_color='grey')
fig.add_shape(type='line', x0=0, y0=0, x1=100, y1=100, line=dict(color='lightgrey', dash='dot'))
fig.update_layout(template='plotly_white', legend_title_text='')
fig.write_html(os.path.join(OUT_DIR, 'node_place.html')); fig.show()

# (b) Top-20 stations: weighted dimension contributions
top = gdf_m.nsmallest(20, 'TOD_rank').sort_values('TOD_index')
fb = go.Figure()
for d in DIM_NAMES:
    fb.add_trace(go.Bar(y=top['name'], x=top[d] * w_equal[d], name=d.replace('_', ' '), orientation='h',
                        hovertemplate='%{y}<br>' + d + ' contribution: %{x:.1f}<extra></extra>'))
fb.update_layout(barmode='stack', template='plotly_white', height=640, title='Top 20 stations - contribution of each 5D dimension to the TOD index',
                 xaxis_title='TOD index (equal weights)')
fb.write_html(os.path.join(OUT_DIR, 'top20_contributions.html')); fb.show()

# (c) Radar profiles: top 3, bottom 1 and median station
pick = list(gdf_m.nsmallest(3, 'TOD_rank')['name']) + list(gdf_m.nlargest(1, 'TOD_rank')['name'])
fr = go.Figure()
for nme in pick:
    r = gdf_m[gdf_m['name'] == nme].iloc[0]
    fr.add_trace(go.Scatterpolar(r=[r[d] for d in DIM_NAMES] + [r[DIM_NAMES[0]]], theta=[d.replace('_', ' ') for d in DIM_NAMES] + [DIM_NAMES[0].replace('_', ' ')],
                                 fill='toself', opacity=.45, name=f"{nme} (#{int(r['TOD_rank'])})"))
fr.update_layout(polar=dict(radialaxis=dict(range=[0, 100])), template='plotly_white', title='5D profile - best three vs. weakest station', height=560)
fr.write_html(os.path.join(OUT_DIR, 'radar_profiles.html')); fr.show()

# (d) Rank stability (Monte-Carlo)
st_ = gdf_m.nsmallest(30, 'TOD_rank').sort_values('TOD_rank')
fs = go.Figure(go.Scatter(x=st_['name'], y=st_['rank_p50'], mode='markers', marker=dict(size=8, color=st_['p_top20'], colorscale='Viridis', colorbar=dict(title='P(top 20%)')),
                          error_y=dict(type='data', symmetric=False, array=st_['rank_p95'] - st_['rank_p50'], arrayminus=st_['rank_p50'] - st_['rank_p05']),
                          hovertemplate='%{x}<br>median rank %{y:.0f}<extra></extra>'))
fs.update_yaxes(autorange='reversed', title='Rank (1 = best)'); fs.update_layout(template='plotly_white', height=520, title='Rank stability of the 30 best stations under 2,000 random weight sets (5-95% interval)')
fs.update_xaxes(tickangle=60); fs.write_html(os.path.join(OUT_DIR, 'rank_stability.html')); fs.show()

## 1️⃣5️⃣ Presentation Dashboard (Summary of Findings)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(18, 14))

# A: coverage by threshold, operational vs full network
x = np.arange(len(cov)); wd = .38
ax[0, 0].bar(x - wd / 2, cov['operational_%'], wd, label='Operational', color='#1f5fbf')
ax[0, 0].bar(x + wd / 2, cov['full_network_%'], wd, label='Phase 1 + 2', color='#f28e2b')
for i, (a, b) in enumerate(zip(cov['operational_%'], cov['full_network_%'])):
    ax[0, 0].text(i - wd / 2, a + 1, f'{a:.0f}%', ha='center', fontsize=9); ax[0, 0].text(i + wd / 2, b + 1, f'{b:.0f}%', ha='center', fontsize=9)
ax[0, 0].set_xticks(x, [f'<= {t} m' for t in cov['threshold_m']]); ax[0, 0].set(title='A: Share of study area within walking distance of a station', ylabel='% of area'); ax[0, 0].legend()

# B: tiers by line
tb = pd.crosstab(gdf_m['line'], gdf_m['TOD_tier'])[list(TIER_COL)]
tb.plot.bar(stacked=True, color=list(TIER_COL.values()), ax=ax[0, 1], edgecolor='k', lw=.4)
ax[0, 1].set(title='B: TOD tiers by line', xlabel='', ylabel='stations'); ax[0, 1].tick_params(axis='x', rotation=20); ax[0, 1].legend(fontsize=8)

# C: node-place classes by status
npc = pd.crosstab(gdf_m['NodePlace'], gdf_m['status'])
npc.plot.barh(stacked=True, ax=ax[1, 0], color=['#1f5fbf', '#f28e2b'], edgecolor='k', lw=.4)
ax[1, 0].set(title='C: Node-Place policy classes', ylabel=''); ax[1, 0].legend(title='')

# D: correlation among dimension scores (+ ridership where available)
cm_cols = DIM_NAMES + ['TOD_index']
sns.heatmap(gdf_m[cm_cols].corr(method='spearman'), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=.5, cbar=False, ax=ax[1, 1])
ax[1, 1].set_title('D: Spearman correlation between 5D scores')
plt.suptitle('Chennai Metro - Accessibility, Catchment and TOD Potential: Summary Dashboard', fontsize=17, fontweight='bold', y=.995)
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/08_summary_dashboard.png'); plt.show()

## 1️⃣6️⃣ Key Findings (computed from the data above)
The cell below fills every number from the analysis, so the narrative always matches the data.

In [ ]:
op_m = gdf_m[gdf_m.operational]; p2_m = gdf_m[~gdf_m.operational]
c800 = cov.loc[cov.threshold_m == 800].iloc[0]
top5 = ', '.join(gdf_m.nsmallest(5, 'TOD_rank')['name'])
mi = moran_tbl.set_index('variable').loc['TOD_index']
n_hot = int(gdf_m['hotspot'].str.startswith('Hot').sum()); n_cold = int(gdf_m['hotspot'].str.startswith('Cold').sum())
cls = gdf_m['NodePlace'].value_counts()

print('=' * 92)
print('KEY FINDINGS')
print('=' * 92)
print(f"1. NETWORK GEOMETRY  - Median same-line station spacing is {gdf_m['nn_same_line_m'].median():,.0f} m "
      f"(operational: {op_m['nn_same_line_m'].median():,.0f} m; Phase 2: {p2_m['nn_same_line_m'].median():,.0f} m).")
print(f"2. COVERAGE          - {c800['operational_%']:.0f}% of the study area lies within 800 m of an operational station; "
      f"Phase 2 lifts this to {c800['full_network_%']:.0f}% (+{c800['gain_pp']:.0f} percentage points).")
print(f"3. CATCHMENT OVERLAP - Summing catchment populations double-counts {100 * (1 - gdf_m['pop_adj'].sum() / gdf_m['population'].sum()):.1f}% of residents; "
      f"{R_DATA} m buffers overlap most where stations are closest.")
print(f"4. TOD POTENTIAL     - Highest-scoring stations: {top5}. "
      f"{int((gdf_m.TOD_tier == 'Tier 1 - Prime').sum())} Tier-1 stations, of which {int(((gdf_m.TOD_tier == 'Tier 1 - Prime') & gdf_m.operational).sum())} are already operational.")
print(f"5. ROBUSTNESS        - Rankings from three weighting schemes agree (min Spearman = {schemes.corr(method='spearman').values.min():.2f}); "
      f"median 5-95% Monte-Carlo rank band = {gdf_m.rank_span.median():.0f} places; {int((gdf_m.p_top20 >= .9).sum())} stations are top-20% in >=90% of draws.")
print(f"6. POLICY CLASSES    - " + '; '.join(f'{k}: {v}' for k, v in cls.items()) + '.')
print(f"7. SPATIAL PATTERN   - Moran's I of TOD index = {mi['Moran' + chr(39) + 's I']:.2f} (p = {mi['p (perm.)']:.3f}) -> {mi['pattern']}; "
      f"{n_hot} hot-spot and {n_cold} cold-spot stations (Gi*, 90%+).")
if HAVE_RIDERSHIP:
    print(f"8. VALIDATION        - TOD index vs weekday boardings: Spearman rho = {rho:.2f} (n = {len(val)}); "
          f"Ridge LOO R2 = {r2_loo:.2f}. Under-performing stations (< -25% vs expectation): {int((val.performance == 'Under-performing').sum())}.")
else:
    print('8. VALIDATION        - skipped (ridership summary not available).')
top_p2 = p2_m.assign(r=gdf_m.loc[p2_m.index, 'priority_rank']).nsmallest(5, 'r')['name'].tolist()
print(f"9. PHASE 2 PRIORITY  - First five stations for TOD planning: {', '.join(top_p2)}.")
print('=' * 92)

## 1️⃣7️⃣ Policy Recommendations by Node–Place Class
| Class | Meaning | Suggested action |
|---|---|---|
| **Mature TOD node** | Strong connectivity *and* intense, walkable surroundings | Protect: cap parking, improve pedestrian/cycle links, maintain land-use mix |
| **Under-built node (intensify)** | Well connected, but surroundings under-use the access | Priority for higher FSI / TDR / value-capture, mixed-use and affordable housing near the station |
| **Under-served place (improve access)** | Dense and active area, weak transit connectivity | Feeder buses, last-mile cycling/walking links, interchange upgrades, wayfinding |
| **Low-intensity periphery (land-bank)** | Limited connectivity and intensity | Land-banking, phased development guidance, reserve right-of-way for feeders |

For **under-performing operational stations** (ridership below built-environment expectation), audit entrance placement, walk-access barriers and feeder services before concluding the catchment is weak.

## 1️⃣8️⃣ Limitations & Data Caveats
1. **Straight-line geometry.** Distances, buffers and Hansen accessibility use Euclidean distance, not walking time on the street network or metro travel time. Street-network effects enter only through the walk-efficiency indicator.
2. **Relative index.** Percentile scoring ranks stations *against each other*; it does not say whether any station meets an absolute TOD standard.
3. **Subjective weights.** Mitigated, not removed, by three weighting schemes and Monte-Carlo sensitivity.
4. **Position quality.** Some Phase 2 coordinates are flagged unreliable or low-confidence in the source directory; their results are indicative.
5. **Uniform-density assumption** inside each buffer for the overlap-adjusted population.
6. **Short ridership window.** Observed boardings cover only a few weeks and ~40 stations; validation is therefore *directional* (n is small, so leave-one-out CV and rank correlations are used instead of a train/test split).
7. **Phase 2 extrapolation.** The demand-potential figure is a relative signal; Phase 2 areas may lie outside the range of the operational training stations.
8. **Spatial tests are exploratory.** Moran's I / Gi* use k = 5 neighbour weights on 134 points without multiple-testing correction.
9. **Correlation is not causation.** The index describes catchment *potential*, not the effect of TOD policy.
10. **Source data.** Built-environment layers derive from OpenStreetMap and JRC GHSL; OSM completeness varies across the city.

## 1️⃣9️⃣ Future Work
* Replace Euclidean buffers with **walking isochrones** from the OSM street graph (`osmnx` / `pandana`).
* Add land-value, FSI/zoning and parcel data to quantify real redevelopment capacity.
* Use travel-time-based accessibility (metro + feeder bus GTFS from CUMTA).
* Extend validation to a longer ridership panel and a mixed-effects / spatial-lag model.

In [ ]:
# ---------- export all results ----------
out_cols = (['station_id', 'name', 'line', 'phase', 'status', 'layout', 'latitude', 'longitude', 'pos_ok', 'nn_dist_m', 'nn_same_line_m',
             'population', 'pop_adj', 'unique_share', 'growth_2030', 'hansen_activity', 'hansen_pop', 'poi_mix_entropy', 'walk_efficiency']
            + DIM_NAMES + ['TOD_index', 'TOD_rank', 'TOD_tier', 'Node', 'Place', 'NodePlace', 'typology',
                           'rank_p05', 'rank_p50', 'rank_p95', 'p_top20', 'gi_z', 'hotspot'])
out_cols += [c for c in ['demand_potential', 'priority_score', 'priority_rank'] if c in gdf_m]
res = gdf_m[out_cols].copy()
res.sort_values('TOD_rank').to_csv(os.path.join(OUT_DIR, 'tod_station_scores.csv'), index=False)
gpd.GeoDataFrame(res, geometry=gdf_m.geometry, crs=CRS_UTM).to_crs(CRS_WGS84).to_file(os.path.join(OUT_DIR, 'tod_station_scores.geojson'), driver='GeoJSON')
buf_tbl.to_csv(os.path.join(OUT_DIR, 'buffer_overlap_summary.csv'), index=False)
cov.to_csv(os.path.join(OUT_DIR, 'coverage_summary.csv'), index=False)
moran_tbl.to_csv(os.path.join(OUT_DIR, 'morans_i_results.csv'), index=False)
weights.to_csv(os.path.join(OUT_DIR, 'tod_weight_schemes.csv'))
if HAVE_RIDERSHIP:
    val[['name', 'line', 'boardings_weekday', 'pred_loo', 'perf_gap_%', 'performance', 'TOD_index']].to_csv(os.path.join(OUT_DIR, 'ridership_validation.csv'), index=False)

print('Saved to', os.path.abspath(OUT_DIR))
for root, _, fs in os.walk(OUT_DIR):
    for f in sorted(fs):
        print('  ', os.path.relpath(os.path.join(root, f), OUT_DIR))
try:
    import shutil
    from google.colab import files
    shutil.make_archive('/content/tod_outputs', 'zip', OUT_DIR)
    files.download('/content/tod_outputs.zip')
except Exception:
    pass

## 2️⃣0️⃣ Viva Voce Preparation Guide
### A. Concept & Motivation
**Q1. What is Transit-Oriented Development and why assess it for Chennai Metro?**
TOD is compact, mixed-use, walkable development concentrated around high-quality transit. It raises ridership, lowers car dependence and lets the public investment in a metro line be recovered through land-value uplift. Chennai's Phase 2 is under construction, so *now* is when station-area planning is cheapest.

**Q2. What is the difference between "transit accessibility", "catchment" and "TOD potential"?**
*Accessibility* = how easily people can reach the station and what the station lets them reach (spacing, coverage, Hansen potential). *Catchment* = the area/people/activities within walking distance of the station. *TOD potential* = how suitable that catchment is for transit-supportive development, summarised by the 5D index.

**Q3. What is the 5D framework?**
Density, Diversity, Design (Cervero & Kockelman), plus Destination accessibility and Distance to transit (Ewing & Cervero). They are the built-environment factors consistently associated with transit use and walking.

### B. GIS Methods
**Q4. Why project to UTM 44N (EPSG:32644)?**
Latitude/longitude are angles; one degree is not a constant length, so buffers and distances in EPSG:4326 are distorted. UTM 44N covers 78°-84° E (Chennai ≈ 80.2° E) and works in metres with very low distortion.

**Q5. How were catchments built and why a 500 m-type radius?**
The dataset's catchment area implies a radius (A = πr²), printed in the notebook as `R_DATA`. 400-800 m is the usual 5-10 minute walking range. Straight-line buffers over-estimate real reach, which the *walk efficiency* indicator (network catchment ÷ circle) partly corrects.

**Q6. How did you handle overlapping catchments?**
Each point inside a buffer is shared equally among the *k* stations that cover it (`unique_share = mean(1/k)`). Adjusted population = population × unique share. This removes double-counting but assumes uniform density in the buffer.

**Q7. How is the coverage surface computed?**
A 250 m grid over the study area (convex hull of all stations + 1.5 km). For each cell a KD-tree gives the distance to the nearest station. Coverage = share of cells within a threshold. Running it for the operational set and the full set gives the Phase 2 gain.

**Q8. What is Hansen accessibility and what are its weaknesses?**
$A_i = \sum_j O_j e^{-\beta d_{ij}}$: opportunities reachable from *i*, discounted with distance. Weaknesses: uses Euclidean not travel-time distance, depends on β (tested with 1.5/3/6 km half-lives; rank correlations printed), and station catchments stand in for the full opportunity surface.

**Q9. What is land-use entropy?**
$H=-\sum_k p_k\ln p_k/\ln K$ over K POI categories. 0 = one use only, 1 = equal shares. It measures mix, not the *quality* or *suitability* of the mix.

### C. Index Construction
**Q10. Why percentile ranks instead of min-max or z-scores?**
They are robust to outliers and skew, need no log transform, are unit-free, and put every indicator on the same 0-100 scale. The cost: they discard magnitude (a tiny gap and a huge gap between neighbours look the same).

**Q11. Why three weight schemes, and what is CRITIC?**
Weights are a judgement call. Equal = neutral base case; Expert = density-heavy planning view; CRITIC = objective weights, giving more weight to dimensions with higher variation and less redundancy with the others (σ × Σ(1−r)). Agreement among schemes (Spearman) shows how much the ranking depends on that choice.

**Q12. What does the Monte-Carlo analysis add?**
It draws 2,000 weight vectors from a Dirichlet centred on equal weights and re-ranks each time. Rank intervals and P(top 20 %) show which stations are robustly strong rather than strong only under one weighting.

**Q13. What is the Node–Place model?**
Bertolini's framework: *Node* = transport connectivity/accessibility, *Place* = intensity of activities in the surroundings. Splitting both axes at the median gives four classes with different policy levers (intensify, improve access, protect, land-bank). It is relative to this network; a "high" score is not an absolute standard.

**Q14. How was the typology chosen?**
K-Means on standardised 5D scores; k from 2-6 selected by silhouette score; clusters are named by dimensions with a standardised mean above 0.35.

### D. Spatial Statistics
**Q15. What does Moran's I tell you here?**
Whether stations with similar TOD scores are near each other. I > 0 with a small permutation p-value ⇒ spatial clustering. It uses row-standardised 5-nearest-neighbour weights; significance from 999 random permutations.

**Q16. Why use Getis-Ord Gi\* as well?**
Moran's I is global; Gi* localises *where* high-value (hot) and low-value (cold) clusters are. It is exploratory here (134 points, no multiple-testing correction).

### E. Validation & Interpretation
**Q17. How did you validate the index and why not a normal train/test split?**
Rank (Spearman) correlation of the index and its dimensions with observed weekday boardings, plus a ridge regression with leave-one-out CV. Only ~40 stations have ridership, so a hold-out set would be too small; LOO uses all data for training in turn.

**Q18. Does a low correlation with ridership mean the index is wrong?**
Not necessarily. Ridership depends on things outside the catchment (network position, service frequency, fares, station age/ramp-up, parking, land-use history). Also the index measures *development potential*, which can exceed current ridership. Hence only mature stations are used and residuals are read as "unrealised demand" or "over-performance", not as errors.

**Q19. How reliable is the Phase 2 priority list?**
Directionally useful, not a forecast: Phase 2 coordinates are partly low-confidence, no ridership exists, and the demand-potential model extrapolates beyond the operational stations. The priority score combines TOD index, 2030 growth and rank stability so it does not hinge on one number.

**Q20. What are the main limitations and how would you improve the study?**
Euclidean distances, relative scoring, subjective weights, short ridership window, OSM completeness. Improvements: walking isochrones from the OSM street graph, GTFS travel-time accessibility, parcel/FSI/land-value data, longer ridership panel and a spatial regression model.

### F. Quick Facts to Memorise
* CRS used: **EPSG:32644** (UTM 44N) for analysis, **EPSG:4326** for storage/web maps, **EPSG:3857** for the static basemap.
* Index = weighted mean of **5 dimension scores**, each the mean of **percentile-ranked** indicators.
* Overlap-adjusted share per station: **mean(1/k)** over points in its buffer.
* All "findings" printed in Section 16 are *computed from the data*, so quote the numbers from your own run.